In [ ]:
import IPython.display
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis

In [ ]:
velocity = na.linspace(-500, 500, axis="wavelength", num=21) * u.km / u.s

In [ ]:
wavelength_rest = 171 * u.AA

In [ ]:
AA = dict(unit=u.AA, equivalencies=u.doppler_optical(wavelength_rest))

In [ ]:
wavelength = velocity.to(**AA)

In [ ]:
position_scene = na.Cartesian2dVectorLinearSpace(
    start=-10 * u.arcsec,
    stop=10 * u.arcsec,
    axis=na.Cartesian2dVectorArray("scene_x", "scene_y"),
    num=na.Cartesian2dVectorArray(64, 64),
)

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 128 + 1, axis="sensor_x") * u.pix,
    y=na.arange(0, 64 + 1, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_scene = na.SpectralPositionalVectorArray(velocity, position_scene)
coordinates_sensor = na.SpectralPositionalVectorArray(velocity, position_sensor)

In [ ]:
scene = ctis.scenes.gaussians(
    inputs=coordinates_scene,
    width=na.SpectralPositionalVectorArray(30 * u.km / u.s, 1 * u.arcsec),
)

In [ ]:
scene = scene + scene.outputs.max() / 100

In [ ]:
coordinates_scene.wavelength = wavelength
coordinates_sensor.wavelength = wavelength

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    colorbar = na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_aspect("equal")
    ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"scene $y$ ({ax.get_ylabel()})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
spectrum = scene.outputs.mean(("scene_x", "scene_y"))

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax2 = ax.twiny()
    na.plt.stairs(
        velocity,
        spectrum,
        ax=ax,
    )
    na.plt.stairs(
        wavelength,
        spectrum,
        ax=ax2
    )
    ax.set_xlabel(f"Doppler velocity ({ax.get_xlabel()})")
    ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    area_effective=1 * u.cm ** 2,
    timedelta_exposure=20 * u.s,
    plate_scale=.4 * u.arcsec / u.pix,
    dispersion=((10 * u.km / u.s).to(**AA) - wavelength_rest) / u.pix,
    angle=na.linspace(0, 360, num=4, axis="channel", endpoint=False) * u.deg,
    wavelength_ref=wavelength_rest,
    position_ref=na.Cartesian2dVectorArray(64, 32) * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    axis_channel="channel",
    axis_wavelength="wavelength",
    axis_scene_xy=("scene_x", "scene_y"),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
images = instrument.image(scene.outputs)

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(
        constrained_layout=True,
        figsize=(9.2, 4),
    )
    norm = plt.Normalize(
        vmin=0,
        vmax=images.outputs.value.ndarray.max(),
    )
    colorizer = plt.Colorizer(
        cmap="gray",
        norm=norm,
    )
    label = "dispersion angle = " + instrument.angle.to_string_array("%03d")
    ani = na.plt.pcolormovie(
        label,
        images.inputs.position.x,
        images.inputs.position.y,
        C=images.outputs.value,
        axis_time="channel",
        ax=ax,
        kwargs_pcolormesh=dict(
            colorizer=colorizer,
        ),
    )
    plt.colorbar(
        mappable=plt.cm.ScalarMappable(colorizer=colorizer),
        ax=ax,
        label=f"signal ({images.outputs.unit:latex_inline})",
    )
    ax.set_aspect("equal")
    ax.set_xlabel(f"sensor $x$ ({images.inputs.position.x.unit})")
    ax.set_ylabel(f"sensor $y$ ({images.inputs.position.y.unit})")

result = ani.to_jshtml(fps=2)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
mart = ctis.inverters.MartInverter(
    instrument=instrument,
    intermediate=True,
)

In [ ]:
guess = 0 * scene.outputs + 1 * scene.outputs.unit

In [ ]:
inversion = mart(
    images=images.outputs,
    guess=guess,
)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        gridspec_kw=dict(width_ratios=[.5, .5, .1]),
        constrained_layout=True,
        figsize=(10, 4.5),
    )
    ax1, ax2, cax = axs
    ax2.set_yticklabels([])
    na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax1,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    label = "iteration = " + inversion.iteration.to_string_array("%d")
    label = label + f"\n{inversion.merit_name} = " + inversion.merit.to_string_array()
    ani, colorbar = na.plt.rgbmovie(
        label,
        scene.inputs.wavelength,
        scene.inputs.position.x,
        scene.inputs.position.y,
        C=inversion.solution,
        axis_time=inversion.inverter.axis_iteration,
        axis_wavelength="wavelength",
        ax=ax2,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax1.set_title("original")
    ax2.set_title("reconstructed")
    unit_x = scene.inputs.position.x.unit
    unit_y = scene.inputs.position.y.unit
    ax1.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax2.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax1.set_ylabel(f"scene $y$ ({unit_y:latex_inline})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
spectrum_inverted = inversion.solution.mean((("scene_x", "scene_y")))
spectrum_inverted = spectrum_inverted[{inversion.inverter.axis_iteration: -1}]

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        wavelength,
        spectrum,
        ax=ax,
        label="original",
    )
    na.plt.stairs(
        wavelength,
        spectrum_inverted,
        ax=ax,
        label="reconstructed",
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")
    ax.legend()